# Absent Edges, Exp A: recompute the central results from episode-level data

This notebook recomputes the paper's central numbers from the per-episode cell tables (`CELLS.jsonl`, one row per
agent episode) of the dataset package **`absent-edges-expA-v1`**. It does not read the paper or its claims registry.
The frozen `ANALYSIS.json` of each run is opened **only to compare**: every recomputed value is printed next to the
frozen one with `MATCH` or `MISMATCH`.

| Section | What is recomputed | Runs |
|---|---|---|
| 1 | Preregistered contrasts H1 = C3 − C2, H2 = C4a − C1, H3 = C1 − C0 (estimate, 95% t interval, t, p, Holm p) | main (Gemma 4 31B BF16) |
| 2 | The same contrasts, H3 compared field by field | 12B, QAT W4A16, hard-omission panel |
| 3 | Omission cost: complete answer (S-VC) minus incomplete answer (block P), seed 11 (exploratory, post hoc) | all four |
| 4 | Completeness claims of every block-P submission | main |
| 5 | Last-response notice A1: deadline − neutral and the list effect under each final-turn line | A1 neutral, deadline and no-line runs |
| 6 | Forest plot of H3 across runs (only if matplotlib is installed) | all four |
| 7 | Summary of matches | |

**Requirements.** Python 3.9 or later, standard library only (matplotlib optional), CPU only, no internet. Run All
takes well under a minute on a laptop (it reads about 70 MB of JSON lines). The notebook writes no files.

**Paths.** Set `ROOT` in the first code cell to the package root, the directory that holds `results/`, `items/` and
`code/`. With the default `"."`, the cell also looks for the package under `/kaggle/input/` (Kaggle attaches datasets
there).

In [ ]:
import importlib.util
import json
import math
import sys
import time
from collections import Counter, defaultdict
from fractions import Fraction
from pathlib import Path

ROOT = "."  # package root of absent-edges-expA-v1 (holds results/, items/, code/)

sys.dont_write_bytecode = True  # the package is read-only: never write __pycache__ into it
T_START = time.time()
MAIN_RUN = "main-20261101b"


def resolve_root(root):
    root = Path(root)
    if (root / "results" / MAIN_RUN / "analysis" / "CELLS.jsonl").is_file():
        return root.resolve()
    kaggle = Path("/kaggle/input")
    for depth in range(1, 5):  # Kaggle mounts a dataset one or more levels below /kaggle/input
        pattern = "/".join(["*"] * depth) + f"/results/{MAIN_RUN}/analysis/CELLS.jsonl"
        for hit in sorted(kaggle.glob(pattern)):
            return hit.parents[3]
    raise FileNotFoundError(f"no results/{MAIN_RUN}/analysis/CELLS.jsonl under {root.resolve()} or /kaggle/input; "
                            "set ROOT to the package root")


ROOT = resolve_root(ROOT)

# Runs recomputed here: label -> run directory under results/
RUNS = {
    "main 31B BF16": "main-20261101b",
    "12B BF16": "repl12b-20261101a",
    "31B QAT W4A16": "qat31b-20261102a",
    "hard panel 31B BF16": "hard-20261102a",
}

# One-sample t statistics (mean, sd, se, t, df, two-sided p, 95% t interval) on exact rationals, standard library
# only. Taken from the package's independent re-analysis script; everything above the family means is written below.
_path = ROOT / "code" / "post-results" / "expA_reanalyze_independent.py"
_spec = importlib.util.spec_from_file_location("expA_reanalyze_independent", _path)
_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_module)
t_statistics = _module.t_statistics

TALLY = Counter()  # MATCH / MISMATCH / SKIPPED over every comparison in this notebook


def elapsed(t0):
    print(f"[section: {time.time() - t0:.1f} s; notebook so far: {time.time() - T_START:.1f} s]")


def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


print("package root:", ROOT)
print("python:", sys.version.split()[0])
for label, run in RUNS.items():
    p = ROOT / "results" / run / "analysis" / "CELLS.jsonl"
    print(f"  {label:22} {run:20} CELLS.jsonl {p.stat().st_size / 1e6:6.1f} MB")

## Method (paper §3, "Outcome and analysis")

Each row of `CELLS.jsonl` is one cell: an (item, block, answer variant, condition, seed) episode with its exit
status and its score. The steps below use only these fields: `block`, `variant`, `condition`, `seed`, `item_id`,
`family`, `cell_state`, `exit_status`, `site_recovered` and (section 4) `complete`.

1. **Cells.** Block P only, seeds 11, 23 and 47. C1 to C4a use the incomplete answer (variant `omit_node`); C0 has no
   answer (variant `none`).
2. **Score.** `site_recovered` (1 if a submitted path and line fall inside a call site of the omitted caller C\*).
   A completed episode that did not submit (exit `NoAnswer`, `ContextLimit` or `FormatExhausted`) scores 0. A cell
   that did not complete (infrastructure) is missing and is left out of its item's mean.
3. **Item mean** over the seeds, per condition; **paired difference** treatment − control within each item.
4. **Family mean** of the item differences (a family is one library).
5. **Estimate** = equal-weight mean over families; two-sided one-sample t test on the family means, df = F − 1, with
   a 95% t interval. Holm step-down across H1, H2 and H3 within each run.

All means are exact fractions, so the estimates can be compared with the frozen `estimate_exact` strings exactly.

In [ ]:
PRIMARY_SEEDS = (11, 23, 47)
ARM_VARIANT = {"C0": "none", "C1": "omit_node", "C2": "omit_node", "C3": "omit_node", "C4a": "omit_node"}
MODEL_EXITS = {"Submitted", "NoAnswer", "ContextLimit", "FormatExhausted"}  # completed model exits
CONTRASTS = [("H1", "C3", "C2"), ("H2", "C4a", "C1"), ("H3", "C1", "C0")]


def score(cell):
    """Recovery of one completed cell: site_recovered if the episode submitted, else 0."""
    return int(cell.get("site_recovered") or 0) if cell["exit_status"] == "Submitted" else 0


def block_p_item_means(cells):
    """condition -> item -> mean score over the primary seeds (Fraction); item -> family; missing-cell counts."""
    scores = defaultdict(lambda: defaultdict(dict))
    family, missing, seen = {}, Counter(), set()
    for c in cells:
        cond = c["condition"]
        if c["block"] != "P" or ARM_VARIANT.get(cond) != c.get("variant") or c["seed"] not in PRIMARY_SEEDS:
            continue
        key = (cond, c["item_id"], c["seed"])
        assert key not in seen, f"duplicate cell {key}"
        seen.add(key)
        family[c["item_id"]] = c["family"]
        if c.get("cell_state") != "completed" or c.get("exit_status") not in MODEL_EXITS:
            missing[cond] += 1
            continue
        scores[cond][c["item_id"]][c["seed"]] = score(c)
    means = {cond: {item: Fraction(sum(by_seed.values()), len(by_seed)) for item, by_seed in items.items()}
             for cond, items in scores.items()}
    return means, family, missing


def family_means(value_by_item, family):
    groups = defaultdict(list)
    for item, value in value_by_item.items():
        groups[family[item]].append(value)
    return {fam: sum(vals, Fraction(0)) / len(vals) for fam, vals in sorted(groups.items())}


def equal_weight(fam_means):
    return sum(fam_means.values(), Fraction(0)) / len(fam_means)


def contrast(means, family, treatment, control):
    paired = sorted(set(means[treatment]) & set(means[control]))
    diffs = {i: means[treatment][i] - means[control][i] for i in paired}
    fam_diff = family_means(diffs, family)
    st = t_statistics(list(fam_diff.values()), 0.95)
    return {
        "estimate": equal_weight(fam_diff),
        "lower": st["lower"], "upper": st["upper"], "t": st["t"], "df": st["df"], "p": st["p_two_sided"],
        "n_families": len(fam_diff), "n_items": len(paired),
        "treatment_rate": equal_weight(family_means({i: means[treatment][i] for i in paired}, family)),
        "control_rate": equal_weight(family_means({i: means[control][i] for i in paired}, family)),
        "family_differences": fam_diff,
    }


def holm(pvalues):
    order = sorted(pvalues, key=lambda k: pvalues[k])
    adjusted, running = {}, 0.0
    for rank, k in enumerate(order):
        running = max(running, min(1.0, (len(order) - rank) * pvalues[k]))
        adjusted[k] = running
    return adjusted


def analyse_run(run):
    cells = read_jsonl(ROOT / "results" / run / "analysis" / "CELLS.jsonl")
    means, family, missing = block_p_item_means(cells)
    res = {hid: contrast(means, family, tre, ctl) for hid, tre, ctl in CONTRASTS}
    for hid, p in holm({hid: r["p"] for hid, r in res.items()}).items():
        res[hid]["holm_p"] = p
    return cells, res, missing


# ---- comparison helpers (frozen values are read only here) ----
REL_TOL, ABS_TOL = 1e-9, 1e-12


def fmt(x):
    if isinstance(x, Fraction):
        return f"{float(x):+.6f} ({x.numerator}/{x.denominator})"
    if isinstance(x, float):
        return f"{x:.6g}"
    return str(x)


def check(name, mine, frozen):
    if frozen is None:
        ok, note = False, "frozen value absent"
    elif isinstance(mine, Fraction) and isinstance(frozen, str):
        ok, note = mine == Fraction(frozen), "exact"
    elif isinstance(mine, (float, Fraction)):
        mine = float(mine)
        diff = abs(mine - float(frozen))
        ok = math.isclose(mine, float(frozen), rel_tol=REL_TOL, abs_tol=ABS_TOL)
        note = f"abs diff {diff:.1e}"
    else:
        ok, note = mine == frozen, "exact"
    TALLY["MATCH" if ok else "MISMATCH"] += 1
    print(f"  {'MATCH   ' if ok else 'MISMATCH'} {name:28} recomputed {fmt(mine):32} frozen {fmt(frozen):28} {note}")
    return ok


def frozen_hypotheses(run):
    doc = json.loads((ROOT / "results" / run / "analysis" / "ANALYSIS.json").read_text(encoding="utf-8"))
    return doc, {h["id"]: h for h in doc["primary"]["hypotheses"]}


def compare_hypothesis(hid, mine, frozen):
    tt = frozen.get("t_test") or {}
    iv = tt.get("interval") or {}
    check(f"{hid} estimate", mine["estimate"], frozen.get("estimate_exact"))
    check(f"{hid} 95% CI lower", mine["lower"], iv.get("lower"))
    check(f"{hid} 95% CI upper", mine["upper"], iv.get("upper"))
    check(f"{hid} t", mine["t"], tt.get("t"))
    check(f"{hid} df", mine["df"], tt.get("df"))
    check(f"{hid} p (two-sided)", mine["p"], tt.get("p_two_sided"))
    check(f"{hid} Holm p", mine["holm_p"], frozen.get("holm_adjusted_p"))
    check(f"{hid} families", mine["n_families"], frozen.get("n_families"))
    check(f"{hid} paired items", mine["n_items"], frozen.get("n_items"))
    check(f"{hid} treatment rate", mine["treatment_rate"], frozen.get("treatment_rate"))
    check(f"{hid} control rate", mine["control_rate"], frozen.get("control_rate"))


def row(label, hid, r):
    return (f"{label:22} {hid}  {100 * float(r['estimate']):+6.1f} pts  95% CI [{100 * r['lower']:+5.1f}, "
            f"{100 * r['upper']:+5.1f}]  t({r['df']}) = {r['t']:6.2f}  p = {r['p']:.2g}  Holm p = {r['holm_p']:.2g}  "
            f"rates {100 * float(r['treatment_rate']):.1f}% vs {100 * float(r['control_rate']):.1f}%  "
            f"({r['n_items']} items, {r['n_families']} families)")


print("defined: block_p_item_means, contrast, holm, analyse_run, check")

## 1. Main run: H1, H2 and H3 (preregistered, confirmatory)

Gemma 4 31B BF16, 750 items in 30 libraries, 11,250 block-P episodes. The preregistration expected H1 and H2 to be
positive and H3 negative.

In [ ]:
t0 = time.time()
RESULTS = {}
main_cells, RESULTS[MAIN_RUN], missing = analyse_run(MAIN_RUN)
print(f"{len(main_cells)} cells read; block-P primary cells missing (not completed): {dict(missing) or 'none'}")
print()
for hid, _, _ in CONTRASTS:
    print(row("main 31B BF16", hid, RESULTS[MAIN_RUN][hid]))

print("\nComparison with the frozen results/main-20261101b/analysis/ANALYSIS.json:")
main_doc, frozen = frozen_hypotheses(MAIN_RUN)
for hid, _, _ in CONTRASTS:
    compare_hypothesis(hid, RESULTS[MAIN_RUN][hid], frozen[hid])
elapsed(t0)

## 2. H3 on the follow-up runs

Each follow-up has its own frozen protocol in `protocol/`: the 12B test (`EXPA_REPL12B.md`, locked before any 12B
main-family outcome), the QAT checkpoint test (`EXPA_QAT31B.md`) and the hard-omission panel (`EXPA_HARD.md`: 236
items in 27 libraries whose omitted caller's call sites do not contain the target's name). The aggregation is the
same as in section 1. H1 and H2 are computed too, because Holm adjusts over all three; only H3 is compared field by
field here. For the hard panel the protocol prespecifies H3 as its single test, so its prespecified p is the
unadjusted two-sided p; the frozen analysis also reports a Holm p over H1 to H3, and both are printed.

In [ ]:
t0 = time.time()
for label, run in RUNS.items():
    if run == MAIN_RUN:
        continue
    t_run = time.time()
    cells, RESULTS[run], missing = analyse_run(run)
    print(f"== {label} ({run}): {len(cells)} cells; block-P primary cells missing: {dict(missing) or 'none'}")
    for hid, _, _ in CONTRASTS:
        print("  " + row(label, hid, RESULTS[run][hid]))
    _, frozen = frozen_hypotheses(run)
    compare_hypothesis("H3", RESULTS[run]["H3"], frozen["H3"])
    print(f"  ({time.time() - t_run:.1f} s)\n")
del cells

print("H3 = C1 - C0 across runs (percentage points):")
for label, run in RUNS.items():
    r = RESULTS[run]["H3"]
    print(f"  {label:22} {100 * float(r['estimate']):+5.1f} [{100 * r['lower']:+5.1f}, {100 * r['upper']:+5.1f}]  "
          f"p = {r['p']:.2g}")
elapsed(t0)

## 3. Omission cost (exploratory, post hoc)

Designed after all results were known (`code/post-results/expA_omission_cost.py`); intervals are nominal. Block S-VC
gives the agent the **complete** answer (variant `vcomplete`, every verified caller listed), at seed 11, under C1
and C3. It differs from the block-P answer (`omit_node`) only in C\*'s entry. Per item: recovery with the complete
answer minus recovery with the incomplete one (seed 11, same condition, completed cells only, a non-submitting
episode scores 0); then family mean; then equal-weight mean over families with a 95% t interval. The paper reports
C1; C3 is shown as well.

The frozen output of the post-results script is `results/post-results/OMISSION_COST.json` in the package. If this
package version does not hold it, the recomputed values are printed and the comparison is marked `SKIPPED`.

In [ ]:
t0 = time.time()
OC_SEED = 11
OC_ARMS = {"incomplete": ("P", "omit_node"), "complete": ("S-VC", "vcomplete")}
OC_CONDITIONS = ("C1", "C3")


def omission_cost(cells):
    rec, sub, family = {}, {}, {}
    for c in cells:
        if c["seed"] != OC_SEED or c.get("cell_state") != "completed" or c.get("exit_status") not in MODEL_EXITS:
            continue
        for arm, (block, variant) in OC_ARMS.items():
            if c["block"] == block and c.get("variant") == variant and c["condition"] in OC_CONDITIONS:
                key = (arm, c["condition"], c["item_id"])
                assert key not in rec, f"duplicate cell {key}"
                rec[key] = score(c)
                sub[key] = int(c["exit_status"] == "Submitted")
                family[c["item_id"]] = c["family"]
    out = {}
    for cond in OC_CONDITIONS:
        items = sorted({k[2] for k in rec if k[1] == cond})
        paired = [i for i in items if ("incomplete", cond, i) in rec and ("complete", cond, i) in rec]

        def fw(values):
            fm = family_means({i: Fraction(v) for i, v in values.items()}, family)
            st = t_statistics(list(fm.values()), 0.95)
            return {"estimate": equal_weight(fm), "lower": st["lower"], "upper": st["upper"], "n_families": len(fm)}

        out[cond] = {
            "items_paired": len(paired), "items_unpaired": len(items) - len(paired),
            "cost": fw({i: rec[("complete", cond, i)] - rec[("incomplete", cond, i)] for i in paired}),
            "recovery_complete": fw({i: rec[("complete", cond, i)] for i in paired}),
            "recovery_incomplete": fw({i: rec[("incomplete", cond, i)] for i in paired}),
            "submitted_complete": fw({i: sub[("complete", cond, i)] for i in paired}),
            "submitted_incomplete": fw({i: sub[("incomplete", cond, i)] for i in paired}),
        }
    return out


OC_FROZEN_PATH = ROOT / "results" / "post-results" / "OMISSION_COST.json"
oc_frozen = json.loads(OC_FROZEN_PATH.read_text(encoding="utf-8")) if OC_FROZEN_PATH.is_file() else None
OC_RUN_KEY = {"main-20261101b": "main", "repl12b-20261101a": "repl12b", "qat31b-20261102a": "qat",
              "hard-20261102a": "hard"}  # run names inside OMISSION_COST.json
OMISSION = {}
for label, run in RUNS.items():
    OMISSION[run] = omission_cost(read_jsonl(ROOT / "results" / run / "analysis" / "CELLS.jsonl"))
    for cond in OC_CONDITIONS:
        v = OMISSION[run][cond]
        c = v["cost"]
        print(f"{label:22} {cond}: recovery {100 * float(v['recovery_incomplete']['estimate']):5.1f}% incomplete -> "
              f"{100 * float(v['recovery_complete']['estimate']):5.1f}% complete; cost "
              f"{100 * float(c['estimate']):+5.1f} pts [{100 * c['lower']:+5.1f}, {100 * c['upper']:+5.1f}]; "
              f"submitted {100 * float(v['submitted_incomplete']['estimate']):.1f}% -> "
              f"{100 * float(v['submitted_complete']['estimate']):.1f}% "
              f"({v['items_paired']} items paired, {v['items_unpaired']} unpaired, {c['n_families']} families)")
        if oc_frozen is None:
            continue
        f = oc_frozen["runs"][OC_RUN_KEY[run]][cond]
        fc = f["omission_cost_complete_minus_incomplete"]
        check(f"{cond} cost estimate", c["estimate"], fc["estimate_exact"])
        check(f"{cond} cost CI lower", c["lower"], fc["ci_low"])
        check(f"{cond} cost CI upper", c["upper"], fc["ci_high"])
        check(f"{cond} recovery complete", v["recovery_complete"]["estimate"],
              f["recovery"]["complete"]["estimate_exact"])
        check(f"{cond} recovery incomplete", v["recovery_incomplete"]["estimate"],
              f["recovery"]["incomplete"]["estimate_exact"])
        check(f"{cond} submitted complete", v["submitted_complete"]["estimate"],
              f["submitted"]["complete"]["estimate_exact"])
        check(f"{cond} submitted incomplete", v["submitted_incomplete"]["estimate"],
              f["submitted"]["incomplete"]["estimate_exact"])
        check(f"{cond} items paired", v["items_paired"], f["items_paired"])
    print()
if oc_frozen is None:
    TALLY["SKIPPED"] += 1
    print(f"SKIPPED: {OC_FROZEN_PATH.relative_to(ROOT)} is not in this package version; values above are recomputed only")
elapsed(t0)

## 4. Completeness claims (main run)

`submit_answer` requires a completeness claim, `yes`, `no` or `unknown`, and every answer block the agent saw stated
completeness "unknown". The cell field `complete` holds the claim of the submitted answer. The tally covers every
block-P submission (exit `Submitted`) of the main run and is compared with the per-condition counts in the frozen
`ANALYSIS.json` (`secondary.rates[block P].completeness_claims`).

The second part counts **certifiably false** claims: a submission that claimed `yes` while missing a
runtime-witnessed verified caller. That flag is not in `CELLS.jsonl`; it is read from the scorer's per-cell rows,
`results/main-20261101b/scores.jsonl` (field `certifiably_false_completeness`).

In [ ]:
t0 = time.time()
claims = defaultdict(Counter)
for c in main_cells:
    if c["block"] == "P" and c.get("exit_status") == "Submitted":
        claims[c["condition"]][str(c.get("complete"))] += 1
total = sum(claims.values(), Counter())
n_sub = sum(total.values())
print(f"main run, block P: {n_sub} submissions; claims {dict(total)}; "
      f"'yes' in {total['yes']} of {n_sub} ({100 * total['yes'] / n_sub:.1f}%)")
for cond in ("C0", "C1", "C2", "C3", "C4a"):
    print(f"  {cond:4} {dict(claims[cond])}")

frozen_rates = {(r["block"], r["condition"]): r for r in main_doc["secondary"]["rates"]}
print("\nComparison with the frozen counts (ANALYSIS.json, secondary.rates):")
for cond in ("C0", "C1", "C2", "C3", "C4a"):
    fz = frozen_rates[("P", cond)]["completeness_claims"]
    for claim in ("yes", "no", "unknown"):
        check(f"{cond} claims '{claim}'", claims[cond][claim], fz.get(claim))
unexpected = set(total) - {"yes", "no", "unknown"}
print("claim values outside yes/no/unknown among submissions:", sorted(unexpected) or "none")

false_claims, submitted = Counter(), Counter()
with open(ROOT / "results" / MAIN_RUN / "scores.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        s = json.loads(line)
        if s.get("block") != "P" or s.get("seed") not in PRIMARY_SEEDS or s.get("exit_status") != "Submitted":
            continue
        if ARM_VARIANT.get(s.get("condition")) != s.get("variant"):
            continue
        submitted[s["condition"]] += 1
        false_claims[s["condition"]] += bool(s.get("certifiably_false_completeness"))
print("\nCertifiably false completeness claims (claimed complete while missing a runtime-witnessed caller):")
for cond in ("C0", "C1", "C2", "C3", "C4a"):
    fz = frozen_rates[("P", cond)]["certifiably_false_completeness"]
    print(f"  {cond:4} {false_claims[cond]} of {submitted[cond]} submissions "
          f"({100 * false_claims[cond] / submitted[cond]:.1f}%)")
    check(f"{cond} false claims", false_claims[cond], fz.get("count"))
    check(f"{cond} submissions", submitted[cond], fz.get("submitted"))
elapsed(t0)

## 5. Last-response notice (A1, preregistered follow-up)

Block P was rerun at seed 11 under three final-turn policies: no line, a neutral line ("Your previous tool call was
processed normally.") and a deadline line ("This is your last response. Call submit_answer now."), appended to the
tool result before the agent's last response (`protocol/EXPA_FINALTURN.md`). The single prespecified test is
deadline − neutral: per item, the mean over C0 and C1 of the score difference (an item enters only with all four
cells), then family means and a one-sample t test over the equal-weight family means. The list effect C1 − C0 is
also recomputed under each line. Frozen values come from `results/finalturn-20261102a/RESULTS.json`.

In [ ]:
t0 = time.time()
FT = {pol: f"finalturn-{pol}-20261102a" for pol in ("neutral", "deadline", "none")}


def seed11_c0c1(run):
    """item -> condition -> score for block P, seed 11, C0 and C1 (completed model exits only); item -> family."""
    out, fam = defaultdict(dict), {}
    for c in read_jsonl(ROOT / "results" / run / "analysis" / "CELLS.jsonl"):
        cond = c["condition"]
        if c["block"] != "P" or c["seed"] != 11 or cond not in ("C0", "C1") or ARM_VARIANT[cond] != c.get("variant"):
            continue
        fam[c["item_id"]] = c["family"]
        if c.get("cell_state") == "completed" and c.get("exit_status") in MODEL_EXITS:
            out[c["item_id"]][cond] = score(c)
    return out, fam


ft = {pol: seed11_c0c1(run) for pol, run in FT.items()}
ft_family = ft["neutral"][1]


def summarise(diffs):
    fam = family_means(diffs, ft_family)
    st = t_statistics(list(fam.values()), 0.95)
    return {"estimate": equal_weight(fam), "lower": st["lower"], "upper": st["upper"], "t": st["t"],
            "df": st["df"], "p": st["p_two_sided"], "n_items": len(diffs), "n_families": len(fam),
            "positive_families": sum(v > 0 for v in fam.values())}


def policy_contrast(a, b):
    sa, sb = ft[a][0], ft[b][0]
    return summarise({i: Fraction(sum(sa[i][c] - sb[i][c] for c in ("C0", "C1")), 2)
                      for i in sa if i in sb and len(sa[i]) == 2 and len(sb[i]) == 2})


def list_effect(pol):
    s = ft[pol][0]
    return summarise({i: Fraction(s[i]["C1"] - s[i]["C0"]) for i in s if len(s[i]) == 2})


frozen_ft = json.loads((ROOT / "results" / "finalturn-20261102a" / "RESULTS.json").read_text(encoding="utf-8"))
prim = policy_contrast("deadline", "neutral")
print(f"Primary, deadline - neutral (mean of C0 and C1): {float(prim['estimate']):+.4f} "
      f"[{prim['lower']:+.4f}, {prim['upper']:+.4f}], t {prim['t']:.3f}, df {prim['df']}, p {prim['p']:.2e}, "
      f"{prim['n_items']} items; positive in {prim['positive_families']} of {prim['n_families']} libraries")
fp = frozen_ft["primary"]
check("A1 primary estimate", prim["estimate"], fp["estimate_exact"])
check("A1 primary ci_low", prim["lower"], fp["ci_low"])
check("A1 primary ci_high", prim["upper"], fp["ci_high"])
check("A1 primary t", prim["t"], fp["t"])
check("A1 primary p", prim["p"], fp["p_two_sided"])
s1 = policy_contrast("deadline", "none")
print(f"S1, deadline - none: {float(s1['estimate']):+.4f} [{s1['lower']:+.4f}, {s1['upper']:+.4f}]")
check("A1 S1 deadline - none", s1["estimate"], frozen_ft["secondary"]["S1_deadline_minus_none"]["estimate_exact"])
print("List effect C1 - C0 under each line:")
for pol in FT:
    le = list_effect(pol)
    fz = frozen_ft["secondary"]["S2_list_effect_by_policy"][pol]
    print(f"  {pol:9} {float(le['estimate']):+.4f} [{le['lower']:+.4f}, {le['upper']:+.4f}]")
    check(f"A1 list effect {pol}", le["estimate"], fz["estimate_exact"])
    check(f"A1 list effect {pol} ci_low", le["lower"], fz["ci_low"])
elapsed(t0)

## 6. H3 across runs

One point per run with its 95% t interval over library means (percentage points). Skipped if matplotlib is not
installed. The hard panel's items are, by construction, ones that name search misses, so its scale differs.

In [ ]:
t0 = time.time()
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib is not installed: plot skipped (the numbers are printed in section 2)")

if plt is not None:
    INK, INK_2, GRID, SERIES = "#0b0b0b", "#52514e", "#e4e3df", "#2a78d6"
    labels = list(RUNS)[::-1]
    fig, ax = plt.subplots(figsize=(7.5, 3.0), dpi=110)
    for y, label in enumerate(labels):
        r = RESULTS[RUNS[label]]["H3"]
        est, lo, hi = 100 * float(r["estimate"]), 100 * r["lower"], 100 * r["upper"]
        ax.plot([lo, hi], [y, y], color=SERIES, linewidth=2, solid_capstyle="round")
        ax.plot(est, y, "o", color=SERIES, markersize=8, markeredgecolor="white", markeredgewidth=2)
        ax.annotate(f"{est:+.1f} [{lo:+.1f}, {hi:+.1f}]", (hi, y), xytext=(8, 0), textcoords="offset points",
                    va="center", fontsize=9, color=INK_2)
    ax.axvline(0, color=INK_2, linewidth=1)
    ax.set_yticks(range(len(labels)), labels, color=INK)
    ax.set_xlim(-2, 30)
    ax.set_xlabel("H3 = C1 - C0: recovery with the incomplete list minus no list (points)", color=INK_2)
    ax.set_title("H3 by run (95% t intervals over library means)", loc="left",
                 fontsize=11, color=INK)
    ax.grid(axis="x", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.tick_params(axis="y", length=0)
    fig.tight_layout()
    plt.show()
elapsed(t0)

## 7. Summary

In [ ]:
print(f"comparisons: {TALLY['MATCH']} MATCH, {TALLY['MISMATCH']} MISMATCH, {TALLY['SKIPPED']} SKIPPED")
print("tolerance: exact fractions and integers must be equal; floats within relative 1e-9 (absolute 1e-12)")
print(f"total run time: {time.time() - T_START:.1f} s")

## Plug in your own agent

The scores above need only one row per episode with the fields listed under "Method". To score another agent on
the same items, run it on the released answers and write such rows; the functions in this notebook then apply
unchanged. What the package documents and contains:

- **Items and their schema.** `items/main/items.jsonl` (schema `expA-item-v2`; 750 targets) and
  `items/hard/items.jsonl` (the hard panel). The field list and the truth definition are in the package
  `README.md`, section 2 ("Item fields", "How truth is defined", "Variants", "Conditions", "Blocks"). The call
  sites of the omitted caller C\* are `V[i].sites` for the entry whose `identity` equals `omitted.identity`; a
  submission recovers C\* when one submitted (path, line) falls inside one of those sites' line spans.
- **What the agent sees.** `dispatch/main/dispatch.jsonl` holds the truth-free view of each item (target, package
  directories, wording pair) and, per variant and condition, the sha256 of the rendered answer cell; the cells
  themselves are in `dispatch/main/payloads/<family>.jsonl` (`sha256`, `bytes`, `payload`). Condition wordings are
  in `items/main/manifest.json` (`wording`) and `protocol/EXPA_PREREG.md`.
- **Episode settings.** Each `runs/<run>/run.json` records the prompts (`prompts`), the tool contract (`tools`,
  including the `submit_answer` envelope with `sites` and `complete`) and the budget (`episode_config`).
  The harness, tools and scorer are in `code/<release>/absent_edges/probe/` (`harness.py`, `tools.py`,
  `score.py`); `PROVENANCE.json` and each run's `run.json` (`code`) say which release tree a run used.
- **Reference episodes.** `runs/<run>/<run>.tar.xz` holds every attempt (`attempt_start.json`, `events.jsonl`,
  `result.json`) with an `INDEX.jsonl` of member hashes; `results/<run>/scores.jsonl` has the scorer's row per cell
  (schema `expA-score-v2`).
- **Model repositories.** The model-visible source trees are not included; rebuild them from the pinned commits in
  `registry/registry.v1.json` and the file manifest `registry/clean-checkouts.json` (package `README.md`,
  section 6).

The package verifier, `python -I -B reproduce/verify.py`, re-runs the frozen analyses byte for byte and checks every
hash chain; this notebook is the shorter, independent path to the headline numbers.